In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger

logger = get_project_logger("Bronze_Layer")
logger.info("Success! The logger is working.")

In [0]:
import time
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from config.datasets import DATASETS

In [0]:
def fetch_batches(ds, limit=5000, start_offset=0, max_rows=75000):
    offset = start_offset
    total_rows = 0

    with requests.Session() as session:
        while total_rows < max_rows:
            start = time.time()

            current_limit = min(limit, max_rows - total_rows)

            params = {
                "resource_id": ds["resource_id"],
                "limit": current_limit,
                "offset": offset
            }

            response = session.get(ds["api_url"], params=params, timeout=60)
            response.raise_for_status()

            data = response.json()
            records = data["result"]["records"]
            batch_rows = len(records)
            total_rows += batch_rows
            
            elapsed = time.time() - start
            logger.info(
                f"[FETCH] dataset={ds['name']} offset={offset} "
                f"rows={batch_rows} took={elapsed:.2f}s"
            )

            if not records:
                logger.info(f"[FETCH] dataset={ds['name']} no more records at offset={offset}")
                break

            yield records
            
            if batch_rows < current_limit:
                break

            offset += limit

In [0]:
def create_df_from_records(records):
    if not records:
        return None

    fields = set()
    for r in records:
        fields.update(r.keys())

    schema = StructType([
        StructField(f, StringType(), True)
        for f in fields
    ])

    logger.info("[BEFORE DF]")
    df = spark.createDataFrame(records, schema=schema)
    logger.info("[AFTER DF]")

    return df

In [0]:
def write_delta(df, ds, mode = "overwrite", max_retries=3):
   
    full_table_name = f"{ds['catalog']}.{ds['schema']}.{ds['table']}"

    for attempt in range(1, max_retries + 1):
        try:
            logger.info(f"[WRITE] attempt={attempt} table={full_table_name} mode={mode}")
            (
                df.write
                .format("delta")
                .mode(mode)
                .saveAsTable(full_table_name)
            )
            logger.info(f"Successfully created/updated table: {full_table_name}")
            return

        except Exception as e:
            logger.warning(f"[WRITE] failed attempt={attempt} table={full_table_name} error={e}")
            if attempt == max_retries:
                logger.error(f"Failed to create/update table: {full_table_name}")
                raise
            time.sleep(10)
    

In [0]:

def ingest_dataset_to_bronze(ds, limit=5000, start_offset=0):
    logger.info(f"\n--- START dataset={ds['name']} ---")

    first_batch = True
    total_rows = 0
    total_batches = 0


    for records in fetch_batches(ds, limit=limit, max_rows=75000):
        if not records:
            logger.info(f"[INGEST] empty batch dataset={ds['name']}")
            continue

        df = create_df_from_records(records)

        if df is None:
            logger.info(f"[INGEST] df is None dataset={ds['name']}")
            continue

        logger.info(f"[BEFORE WRITE] dataset={ds['name']} batch={total_batches+1}")

        mode = "overwrite" if first_batch else "append"
        write_delta(df, ds, mode=mode)
        logger.info(f"[AFTER WRITE] dataset={ds['name']} batch={total_batches+1}")

        batch_rows = len(records)
        total_rows += batch_rows
        total_batches += 1
        first_batch = False

        logger.info(
            f"[PROGRESS] dataset={ds['name']} "
            f"batch={total_batches} batch_rows={batch_rows} total_rows={total_rows}"
        )

    if total_batches == 0:
        logger.info(f"[INGEST] no data found for dataset={ds['name']}")
    else:
        logger.info(
            f"--- END dataset={ds['name']} batches={total_batches} total_rows={total_rows} ---"
        )

In [0]:
for ds in DATASETS:
    if not ds.get("enabled", True):
        continue

    ingest_dataset_to_bronze(ds, limit=5000)

logger.info("Bronze layer ingestion completed")